# Dataset Transformation and Feature Engineering P1
### Previous Step: Data Imputation (Missing Values Filled)
### Next Step: Modeling Setup
---
##### TODO:
    - Correcting feature names for correct interpretation
    - Parsing date information for future features ease of creation
    - Add all auto-regressive features (lags, statistical methods)
    - Temporal Feature Correlation Analysis
    - Feature Behavioral Analysis
    - Feature Engineering Design
    - Temporal Features Refinement and Exportation


This is based on [Feature & Target Variable Document](../docs/FE_logs/week7_feature_target.md)

In [248]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import holidays # To get holidays in Mexico for feature engineering

# Cargar datos imputados (la base del feature engineering)
df_pre_transformed = pd.read_excel(
    "../data/processed/remissions_db_imputed_all_plants_2025.xlsx",
)

print(f"Rows: {len(df_pre_transformed):,}")
print(f"Columns: {df_pre_transformed.columns.tolist()}")
print(f"Date range: {df_pre_transformed['order_date'].min()} to {df_pre_transformed['order_date'].max()}")
print(f"\nPlants: {df_pre_transformed['ship_plant_code'].unique()}")
df_pre_transformed.head()


Rows: 366,961
Columns: ['tkt_code', 'order_date', 'start_time', 'truck_code', 'ship_plant_code', 'u_Volumen', 'typed_time', 'at_plant_time', 'u_Cicle', 'name', 'Nombre del proyecto', 'ship_addr_line', 'map_page', 'hour', 'day_of_week', 'hour_bucket', 'month', 'imputed_source_plant', 'is_imputed', 'imputed_method', 'imputed_range', 'source_plant_code']
Date range: 2020-02-04 00:00:00 to 2026-04-24 00:00:00

Plants: [512 510 511 710 515 514]


,tkt_code,order_date,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,...,map_page,hour,day_of_week,hour_bucket,month,imputed_source_plant,is_imputed,imputed_method,imputed_range,source_plant_code
0,51253281.0,2020-02-04,2020-02-04 05:00:00,7043.0,512,6.0,2020-02-04 05:30:26,2020-02-04 06:50:05,80,COPACHISA,...,CH-F3,5,1,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,51253282.0,2020-02-04,2020-02-04 05:00:00,4366.0,512,6.0,2020-02-04 05:31:47,2020-02-04 06:44:47,73,COPACHISA,...,CH-F3,5,1,NaT,NaN,NaN,NaN,NaN,NaN,NaN
2,51253307.0,2020-02-04,2020-02-04 05:00:00,4360.0,512,5.0,2020-02-04 11:17:49,2020-02-04 12:17:18,60,COPACHISA,...,CH-F3,5,1,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,51013232.0,2020-02-04,2020-02-04 07:00:00,7044.0,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,...,CH-F1,7,1,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,51165724.0,2020-02-04,2020-02-04 07:00:00,9431.0,511,4.0,2020-02-04 07:01:34,2020-02-04 08:46:44,105,PARCELAS CHUVISCAR,...,CH-V1,7,1,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [249]:
# Drop last section helper columns

df_pre_transformed = df_pre_transformed.drop(columns=["hour", "day_of_week", "hour_bucket","month", "truck_code","imputed_source_plant","imputed_range","imputed_method","source_plant_code"], errors="ignore")

In [250]:
print(f"Columns: {df_pre_transformed.columns.tolist()}")


Columns: ['tkt_code', 'order_date', 'start_time', 'ship_plant_code', 'u_Volumen', 'typed_time', 'at_plant_time', 'u_Cicle', 'name', 'Nombre del proyecto', 'ship_addr_line', 'map_page', 'is_imputed']


In [251]:
# Change name for all columns

df_pre_transformed = df_pre_transformed.rename(columns={
    "order_date": "remission_date",
    "ship_plant_code": "plant_code",
    "start_time": "programmed_departure_time",
    "typed_time": "actual_departure_time",
    "name": "client_name",
    "Nombre del proyecto": "project_name",
    "u_Cicle": "truck_cycle_minutes"
})

Parsing y limpieza de columnas

In [252]:
# Parse remission date to new columns = [month_of_year, day_of_week, day_of_month]

df_pre_transformed["month_of_year"] = df_pre_transformed["remission_date"].dt.month
df_pre_transformed["day_of_week"] = df_pre_transformed["remission_date"].dt.dayofweek
df_pre_transformed["day_of_month"] = df_pre_transformed["remission_date"].dt.day

df_pre_transformed.drop(columns=["remission_date"], inplace=True)
df_pre_transformed.drop(columns=["tkt_code"], inplace=True)

print(f"Columns: {df_pre_transformed.columns.tolist()}")
df_pre_transformed.head()

Columns: ['programmed_departure_time', 'plant_code', 'u_Volumen', 'actual_departure_time', 'at_plant_time', 'truck_cycle_minutes', 'client_name', 'project_name', 'ship_addr_line', 'map_page', 'is_imputed', 'month_of_year', 'day_of_week', 'day_of_month']


,programmed_departure_time,plant_code,u_Volumen,actual_departure_time,at_plant_time,truck_cycle_minutes,client_name,project_name,ship_addr_line,map_page,is_imputed,month_of_year,day_of_week,day_of_month
0,2020-02-04 05:00:00,512,6.0,2020-02-04 05:30:26,2020-02-04 06:50:05,80,COPACHISA,INACTIVA CESSNA SUPRA,AV DEMING PARQUE INDUSTRIAL SUPRA ENSEG,CH-F3,NaN,2,1,4
1,2020-02-04 05:00:00,512,6.0,2020-02-04 05:31:47,2020-02-04 06:44:47,73,COPACHISA,INACTIVA CESSNA SUPRA,AV DEMING PARQUE INDUSTRIAL SUPRA ENSEG,CH-F3,NaN,2,1,4
2,2020-02-04 05:00:00,512,5.0,2020-02-04 11:17:49,2020-02-04 12:17:18,60,COPACHISA,INACTIVA CESSNA SUPRA,AV DEMING PARQUE INDUSTRIAL SUPRA ENSEG,CH-F3,NaN,2,1,4
3,2020-02-04 07:00:00,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1,NaN,2,1,4
4,2020-02-04 07:00:00,511,4.0,2020-02-04 07:01:34,2020-02-04 08:46:44,105,PARCELAS CHUVISCAR,LUIS DERMA FRACC MAGENTA FASE 2,LUIS DERMA FRACC MAGENTA FASE 2 LUIS,CH-V1,NaN,2,1,4


In [253]:
# Export for feature engineering p2
df_pre_transformed.to_excel("../data/processed/pre_transformed_for_FE_p2.xlsx", index=False)

In [254]:
# Delete spatial variables that won't be used in the first iteration
df_pre_transformed = df_pre_transformed.drop(columns=['map_page', 'ship_addr_line', 'project_name', 'truck_cycle_minutes', 'client_name'])

In [255]:
# Floor programmed_departure_time to the hour (e.g. 7:45am -> 7:00am)
df_pre_transformed["programmed_departure_time"] = df_pre_transformed["programmed_departure_time"].dt.floor('h')

In [256]:
df_pre_transformed.head()

,programmed_departure_time,plant_code,u_Volumen,actual_departure_time,at_plant_time,is_imputed,month_of_year,day_of_week,day_of_month
0,2020-02-04 05:00:00,512,6.0,2020-02-04 05:30:26,2020-02-04 06:50:05,NaN,2,1,4
1,2020-02-04 05:00:00,512,6.0,2020-02-04 05:31:47,2020-02-04 06:44:47,NaN,2,1,4
2,2020-02-04 05:00:00,512,5.0,2020-02-04 11:17:49,2020-02-04 12:17:18,NaN,2,1,4
3,2020-02-04 07:00:00,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,NaN,2,1,4
4,2020-02-04 07:00:00,511,4.0,2020-02-04 07:01:34,2020-02-04 08:46:44,NaN,2,1,4


In [257]:
# Drop actual_departure_time and at_plant_time
df_pre_transformed.drop(columns=["actual_departure_time", "at_plant_time"], inplace=True)

In [258]:
# print column names
df_pre_transformed.columns.tolist()

['programmed_departure_time',
 'plant_code',
 'u_Volumen',
 'is_imputed',
 'month_of_year',
 'day_of_week',
 'day_of_month']

#### DATASET FORMAT TRANSFORMATION

- `Long format`: TFT and other models will require a long format to capture all demand changes to each plant






In [259]:
# Transform to long format, aggregating by programmed_departure_time and plant_code
df_long = df_pre_transformed.groupby(
    ["programmed_departure_time", "plant_code"]
).agg( 
    volume_m3=("u_Volumen", "sum"),           # sum of volume per hour
    remission_count=("u_Volumen", "count"),   # count of remissions for that hour
    is_imputed=("is_imputed", "max"),         # 1 if a remission was imputed
    month_of_year=("month_of_year", "first"), # month of year
    day_of_week=("day_of_week", "first"),     # day of week
    day_of_month=("day_of_month", "first"),   # day of month
).reset_index()

print(f"Filas antes: {len(df_pre_transformed):,}")
print(f"Filas después (long format): {len(df_long):,}")
df_long.head()

Filas antes: 366,961
Filas después (long format): 87,810


,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month
0,2020-02-04 05:00:00,512,17.0,3,NaN,2,1,4
1,2020-02-04 07:00:00,510,11.5,2,NaN,2,1,4
2,2020-02-04 07:00:00,511,21.0,5,NaN,2,1,4
3,2020-02-04 07:00:00,710,8.0,2,NaN,2,1,4
4,2020-02-04 08:00:00,510,15.0,4,NaN,2,1,4


In [260]:
# extract hour from programmed_departure_time
df_long["programmed_departure_hour"] = df_long["programmed_departure_time"].dt.hour
df_long.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour
0,2020-02-04 05:00:00,512,17.0,3,NaN,2,1,4,5
1,2020-02-04 07:00:00,510,11.5,2,NaN,2,1,4,7
2,2020-02-04 07:00:00,511,21.0,5,NaN,2,1,4,7
3,2020-02-04 07:00:00,710,8.0,2,NaN,2,1,4,7
4,2020-02-04 08:00:00,510,15.0,4,NaN,2,1,4,8


Keeping programmed_departure_hour as datetime timestamp for FE

In [261]:
# Missed year feature, no need for encoding tho (lightgbm update)
df_long["year"] = df_long["programmed_departure_time"].dt.year

In [262]:
# Complete range per plant
plants = df_long["plant_code"].unique()
full_index = []

for plant in plants:
    plant_data = df_long[df_long["plant_code"] == plant]
    start = plant_data["programmed_departure_time"].min()
    end = plant_data["programmed_departure_time"].max()
    full_hours = pd.date_range(start, end, freq="h")
    full_index.extend([(h, plant) for h in full_hours])

full_index = pd.MultiIndex.from_tuples(full_index, names=["programmed_departure_time", "plant_code"])

df_transformed = df_long.set_index(["programmed_departure_time", "plant_code"]).reindex(full_index).reset_index()

df_transformed["volume_m3"] = df_transformed["volume_m3"].fillna(0)
df_transformed["remission_count"] = df_transformed["remission_count"].fillna(0)

df_transformed["hour"] = df_transformed["programmed_departure_time"].dt.hour
df_transformed["day_of_week"] = df_transformed["programmed_departure_time"].dt.dayofweek
df_transformed["month_of_year"] = df_transformed["programmed_departure_time"].dt.month
df_transformed["day_of_month"] = df_transformed["programmed_departure_time"].dt.day
df_transformed["year"] = df_transformed["programmed_departure_time"].dt.year

df_transformed = df_transformed.sort_values(["plant_code", "programmed_departure_time"]).reset_index(drop=True)

In [263]:
df_transformed.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,hour
0,2020-02-04 07:00:00,510,11.5,2.0,NaN,2,1,4,7.0,2020,7
1,2020-02-04 08:00:00,510,15.0,4.0,NaN,2,1,4,8.0,2020,8
2,2020-02-04 09:00:00,510,20.0,5.0,NaN,2,1,4,9.0,2020,9
3,2020-02-04 10:00:00,510,21.5,7.0,NaN,2,1,4,10.0,2020,10
4,2020-02-04 11:00:00,510,32.0,9.0,NaN,2,1,4,11.0,2020,11


In [264]:
# Preventive cleaning re-process
df_transformed["programmed_departure_time"] = pd.to_datetime(df_transformed["programmed_departure_time"])

# Re-calculate hour
df_transformed["programmed_departure_hour"] = df_transformed["programmed_departure_time"].dt.hour

# Fix: NaN to 0 for is_imputed
df_transformed["is_imputed"] = df_transformed["is_imputed"].fillna(0).astype(int)

# Verificar
print("NaN values:")
print(df_transformed.isna().sum())
print("Data types:")
print(df_transformed.dtypes)

NaN values:
programmed_departure_time    0
plant_code                   0
volume_m3                    0
remission_count              0
is_imputed                   0
month_of_year                0
day_of_week                  0
day_of_month                 0
programmed_departure_hour    0
year                         0
hour                         0
dtype: int64
Data types:
programmed_departure_time    datetime64[us]
plant_code                            int64
volume_m3                           float64
remission_count                     float64
is_imputed                            int64
month_of_year                         int32
day_of_week                           int32
day_of_month                          int32
programmed_departure_hour             int32
year                                  int32
hour                                  int32
dtype: object


In [265]:
df_transformed.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,hour
0,2020-02-04 07:00:00,510,11.5,2.0,0,2,1,4,7,2020,7
1,2020-02-04 08:00:00,510,15.0,4.0,0,2,1,4,8,2020,8
2,2020-02-04 09:00:00,510,20.0,5.0,0,2,1,4,9,2020,9
3,2020-02-04 10:00:00,510,21.5,7.0,0,2,1,4,10,2020,10
4,2020-02-04 11:00:00,510,32.0,9.0,0,2,1,4,11,2020,11


In [266]:
# Save transformed dataset for record management
df_transformed.to_excel("../data/processed/transformed_dataset_pre_FE.xlsx", index=False)

### FEATURE ENGINEERING

Have-to-go features:
- `is_quincena` (from last period)
- `is_holiday` (from last period, must analyze again)
- `was_open` (from **cleaning_phase**)
- `is_sunday`, `is_saturday` (from **cleaning_phase**)
- `is_imputed` (from **data_imputation** phase)
- `statistic lags` (mentioned in **week7_feature_target**)
- `time-regressive lags` (mentioned in **week7_feature_target**)
- `cycling encoding` transformed timestamps

In [267]:
# From here I need you to do the correlation analysis to justify each have-to-go feature (because it is in the hourly_features.parquet) for the model training dataset

# I need small chunks of code per cell, doing simple correlation to find out regression significance towards the target variable (volume_m3 or remission_count)

# If you can make it look less like AI and more like a human (maybe grounding in previous notebooks) doing the analysis, that would be great.

# Always add a comment at the top of each cell explaining what the code is doing and why, as if you were a data scientist exploring the data and justifying your feature selection for a regression model.

In [268]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Load the transformed dataset
df_pre_FE = pd.read_excel("../data/processed/transformed_dataset_pre_FE.xlsx")

print(f"Shape: {df_pre_FE.shape}")
print(f"\nColumns: {df_pre_FE.columns.tolist()}")
print(f"\nData types:\n{df_pre_FE.dtypes}")
print(f"\nNaNs per column:\n{df_pre_FE.isna().sum()}")
print(f"\nFirst 5 rows:")
df_pre_FE.head()

Shape: (305521, 11)

Columns: ['programmed_departure_time', 'plant_code', 'volume_m3', 'remission_count', 'is_imputed', 'month_of_year', 'day_of_week', 'day_of_month', 'programmed_departure_hour', 'year', 'hour']

Data types:
programmed_departure_time    datetime64[us]
plant_code                            int64
volume_m3                           float64
remission_count                       int64
is_imputed                            int64
month_of_year                         int64
day_of_week                           int64
day_of_month                          int64
programmed_departure_hour             int64
year                                  int64
hour                                  int64
dtype: object

NaNs per column:
programmed_departure_time    0
plant_code                   0
volume_m3                    0
remission_count              0
is_imputed                   0
month_of_year                0
day_of_week                  0
day_of_month                 0
programmed

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,hour
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,7
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,8
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,9
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,10
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,11


In [269]:
# Cell 2: Time index and cyclic encoding

# 1. Ensure sorted by plant and time, then create sequential index
df_pre_FE = df_pre_FE.sort_values(["plant_code", "programmed_departure_time"]).reset_index(drop=True)
df_pre_FE["time_idx"] = df_pre_FE.groupby("plant_code").cumcount()

# 2. Cyclic encoding: hour (0-23) -> sin/cos
df_pre_FE["hour_sin"] = np.sin(2 * np.pi * df_pre_FE["programmed_departure_hour"] / 24)
df_pre_FE["hour_cos"] = np.cos(2 * np.pi * df_pre_FE["programmed_departure_hour"] / 24)

# 3. Cyclic encoding: day_of_week (0-6) -> sin/cos
df_pre_FE["day_week_sin"] = np.sin(2 * np.pi * df_pre_FE["day_of_week"] / 7)
df_pre_FE["day_week_cos"] = np.cos(2 * np.pi * df_pre_FE["day_of_week"] / 7)

# 4. Cyclic encoding: month (1-12) -> sin/cos
# We use (month - 1) so January=0, December=11
df_pre_FE["month_sin"] = np.sin(2 * np.pi * (df_pre_FE["month_of_year"] - 1) / 12)
df_pre_FE["month_cos"] = np.cos(2 * np.pi * (df_pre_FE["month_of_year"] - 1) / 12)

print("New columns:")
print([c for c in df_pre_FE.columns if c not in ['programmed_departure_time', 'plant_code', 'volume_m3', 
      'remission_count', 'is_imputed', 'month_of_year', 'day_of_week', 'day_of_month', 
      'programmed_departure_hour', 'year']])

# Plot: hours on a circle (sample for speed)
sample = df_pre_FE[df_pre_FE["plant_code"] == 510].sample(min(3000, len(df_pre_FE[df_pre_FE["plant_code"] == 510])))

fig = px.scatter(
    sample,
    x="hour_sin",
    y="hour_cos",
    color="programmed_departure_hour",
    title="Cyclic Encoding: Hours Mapped to a Circle (Plant 510 sample)",
    labels={"hour_sin": "sin(2π·hour/24)", "hour_cos": "cos(2π·hour/24)", "programmed_departure_hour": "Hour"},
    color_continuous_scale="Viridis"
)
fig.update_layout(width=600, height=600)
fig.show()

New columns:
['hour', 'time_idx', 'hour_sin', 'hour_cos', 'day_week_sin', 'day_week_cos', 'month_sin', 'month_cos']


In [270]:
df_pre_FE.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,hour,time_idx,hour_sin,hour_cos,day_week_sin,day_week_cos,month_sin,month_cos
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,7,0,0.965926,-0.258819,0.781831,0.62349,0.5,0.866025
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,8,1,0.866025,-0.500000,0.781831,0.62349,0.5,0.866025
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,9,2,0.707107,-0.707107,0.781831,0.62349,0.5,0.866025
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,10,3,0.500000,-0.866025,0.781831,0.62349,0.5,0.866025
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,11,4,0.258819,-0.965926,0.781831,0.62349,0.5,0.866025


In [271]:
# Using Pearson correlation to check linear relationships between features and target (volume_m3)

# Cell 3: Correlation check — cyclic & raw temporal features vs. volume_m3
# After creating sine/cosine encodings, I want to see which temporal signals
# actually track with concrete demand. Weak correlations don't necessarily mean
# discard (tree models capture non-linear stuff), but strong correlations give
# me confidence these features belong in the model.

features_to_check = [
    "programmed_departure_hour", "day_of_week", "month_of_year",
    "hour_sin", "hour_cos",
    "day_week_sin", "day_week_cos",
    "month_sin", "month_cos"
]

# Pearson correlation with target
corrs = df_pre_FE[features_to_check + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")

# Plot
fig = go.Figure()
colors = ["#1f77b4" if ("sin" in f or "cos" in f) else "#ff7f0e" for f in corrs.index]
fig.add_trace(go.Bar(x=corrs.index, y=corrs.values, marker_color=colors))
fig.update_layout(
    title="Correlation with volume_m3: Raw vs. Cyclic Temporal Features",
    xaxis_tickangle=-45,
    yaxis_title="Pearson r",
    template="plotly_white",
    width=700, height=450
)
fig.show()

print("Correlations (sorted):")
print(corrs.sort_values(ascending=False).round(3))

Correlations (sorted):
hour_sin                     0.121
day_week_sin                 0.098
month_of_year                0.027
month_sin                   -0.011
month_cos                   -0.021
programmed_departure_hour   -0.029
day_of_week                 -0.096
day_week_cos                -0.113
hour_cos                    -0.418
Name: volume_m3, dtype: float64


In [272]:
# Cell 4: Correlation on non-zero hours only
# The zeros are drowning the signal. Let's filter to hours where there WAS demand
# and see if temporal features actually track with HOW MUCH demand.

df_positive = df_pre_FE[df_pre_FE["volume_m3"] > 0].copy()

features_to_check = [
    "programmed_departure_hour", "day_of_week", "month_of_year",
    "hour_sin", "hour_cos",
    "day_week_sin", "day_week_cos",
    "month_sin", "month_cos"
]

corrs_zero = df_pre_FE[features_to_check + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")
corrs_positive = df_positive[features_to_check + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")

# Compare side by side
comparison = pd.DataFrame({
    "all_hours": corrs_zero,
    "volume_gt_0": corrs_positive
})

fig = go.Figure()
fig.add_trace(go.Bar(name="All hours (incl. zeros)", x=comparison.index, y=comparison["all_hours"], marker_color="#ff7f0e"))
fig.add_trace(go.Bar(name="Volume > 0 only", x=comparison.index, y=comparison["volume_gt_0"], marker_color="#1f77b4"))
fig.update_layout(
    title="Correlation with volume_m3: All hours vs. Hours with demand only",
    barmode="group",
    xaxis_tickangle=-45,
    yaxis_title="Pearson r",
    template="plotly_white",
    width=800, height=450
)
fig.show()

print(comparison.round(3))

                           all_hours  volume_gt_0
programmed_departure_hour     -0.029       -0.173
day_of_week                   -0.096        0.076
month_of_year                  0.027        0.021
hour_sin                       0.121        0.164
hour_cos                      -0.418        0.017
day_week_sin                   0.098       -0.015
day_week_cos                  -0.113       -0.080
month_sin                     -0.011       -0.008
month_cos                     -0.021        0.004


In [273]:
df_pre_FE.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,hour,time_idx,hour_sin,hour_cos,day_week_sin,day_week_cos,month_sin,month_cos
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,7,0,0.965926,-0.258819,0.781831,0.62349,0.5,0.866025
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,8,1,0.866025,-0.500000,0.781831,0.62349,0.5,0.866025
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,9,2,0.707107,-0.707107,0.781831,0.62349,0.5,0.866025
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,10,3,0.500000,-0.866025,0.781831,0.62349,0.5,0.866025
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,11,4,0.258819,-0.965926,0.781831,0.62349,0.5,0.866025


In [274]:
# Cell 5: Weekend analysis — checking if weekends actually affect demand
# Business logic says plants are closed Sundays and half-day Saturdays.
# If this is true, we should see massive volume drops on day_of_week = 5,6.
# Let's build the features and check.

df_pre_FE["is_weekend"] = (df_pre_FE["day_of_week"] >= 5).astype(int)
df_pre_FE["is_sunday"] = (df_pre_FE["day_of_week"] == 6).astype(int)

# Average volume by day of week
daily_avg = df_pre_FE.groupby("day_of_week")["volume_m3"].mean().reset_index()
daily_avg["day_name"] = daily_avg["day_of_week"].map({
    0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"
})

fig = px.bar(
    daily_avg,
    x="day_name",
    y="volume_m3",
    title="Average Hourly Volume by Day of Week",
    labels={"volume_m3": "Avg Volume (m³/hr)", "day_name": "Day"},
    color="volume_m3",
    color_continuous_scale="RdYlGn"
)
fig.update_layout(width=600, height=400, template="plotly_white")
fig.show()

# Correlation check
print("Correlation with volume_m3:")
print(f"is_weekend: {df_pre_FE['is_weekend'].corr(df_pre_FE['volume_m3']):.3f}")
print(f"is_sunday:  {df_pre_FE['is_sunday'].corr(df_pre_FE['volume_m3']):.3f}")

Correlation with volume_m3:
is_weekend: -0.154
is_sunday:  -0.169


In [275]:
# Cell 6: Sunday effect during business hours only
# If we filter to hours 7-17, Sundays should be VERY different from weekdays.
# This shows the TRUE business impact of Sunday closure.

business_hours = df_pre_FE[df_pre_FE["programmed_departure_hour"].between(7, 17)].copy()

daily_avg_business = business_hours.groupby("day_of_week")["volume_m3"].mean().reset_index()
daily_avg_business["day_name"] = daily_avg_business["day_of_week"].map({
    0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"
})

fig = px.bar(
    daily_avg_business,
    x="day_name",
    y="volume_m3",
    title="Average Volume by Day (Business Hours Only: 7am–5pm)",
    labels={"volume_m3": "Avg Volume (m³/hr)", "day_name": "Day"},
    color="volume_m3",
    color_continuous_scale="RdYlGn"
)
fig.update_layout(width=600, height=400, template="plotly_white")
fig.show()

print("Correlation during business hours:")
print(f"is_sunday: {business_hours['is_sunday'].corr(business_hours['volume_m3']):.3f}")
print(f"is_weekend: {business_hours['is_weekend'].corr(business_hours['volume_m3']):.3f}")

print("\nAverage volume by day (business hours):")
print(daily_avg_business.set_index("day_name")["volume_m3"].round(2))

Correlation during business hours:
is_sunday: -0.303
is_weekend: -0.277

Average volume by day (business hours):
day_name
Mon     7.94
Tue     9.42
Wed     9.18
Thu    11.48
Fri    11.99
Sat     6.67
Sun     0.09
Name: volume_m3, dtype: float64


is_sunday and is_weekend justified due to actual correlation when analyzed Pearson on business hours

In [276]:
df_pre_FE.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,hour,time_idx,hour_sin,hour_cos,day_week_sin,day_week_cos,month_sin,month_cos,is_weekend,is_sunday
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,7,0,0.965926,-0.258819,0.781831,0.62349,0.5,0.866025,0,0
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,8,1,0.866025,-0.500000,0.781831,0.62349,0.5,0.866025,0,0
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,9,2,0.707107,-0.707107,0.781831,0.62349,0.5,0.866025,0,0
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,10,3,0.500000,-0.866025,0.781831,0.62349,0.5,0.866025,0,0
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,11,4,0.258819,-0.965926,0.781831,0.62349,0.5,0.866025,0,0


In [277]:
# Cell 7: Quincena analysis — testing if payday cycles affect concrete demand
# Business intuition: more disposable income around quincena = more construction starts.
# If true, we should see volume spikes around the 15th and last day of each month.

def days_to_quincena(d):
    """Days until next payday (15th or last day of month)"""
    if d.day <= 15:
        return 15 - d.day
    else:
        last_day = pd.Timestamp(d.year, d.month, 1) + pd.offsets.MonthEnd(0)
        return (last_day - pd.Timestamp(d)).days

# Apply to the datetime
df_pre_FE["days_to_quincena"] = df_pre_FE["programmed_departure_time"].apply(days_to_quincena)
df_pre_FE["is_quincena"] = ((df_pre_FE["day_of_month"] == 15) | 
                            ((df_pre_FE["programmed_departure_time"] + pd.Timedelta(days=1)).dt.day == 1)).astype(int)

# Average volume by days_to_quincena
quincena_avg = df_pre_FE.groupby("days_to_quincena")["volume_m3"].mean().reset_index()

fig = px.line(
    quincena_avg,
    x="days_to_quincena",
    y="volume_m3",
    title="Average Volume by Days to Next Payday (Quincena)",
    labels={"days_to_quincena": "Days to quincena", "volume_m3": "Avg Volume (m³/hr)"}
)
fig.update_layout(width=700, height=400, template="plotly_white")
fig.show()

print("Correlation with volume_m3:")
print(f"days_to_quincena: {df_pre_FE['days_to_quincena'].corr(df_pre_FE['volume_m3']):.3f}")
print(f"is_quincena:      {df_pre_FE['is_quincena'].corr(df_pre_FE['volume_m3']):.3f}")

Correlation with volume_m3:
days_to_quincena: -0.009
is_quincena:      -0.006


In [278]:
df_pre_FE.head()    

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,...,hour_sin,hour_cos,day_week_sin,day_week_cos,month_sin,month_cos,is_weekend,is_sunday,days_to_quincena,is_quincena
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,...,0.965926,-0.258819,0.781831,0.62349,0.5,0.866025,0,0,11,0
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,...,0.866025,-0.500000,0.781831,0.62349,0.5,0.866025,0,0,11,0
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,...,0.707107,-0.707107,0.781831,0.62349,0.5,0.866025,0,0,11,0
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,...,0.500000,-0.866025,0.781831,0.62349,0.5,0.866025,0,0,11,0
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,...,0.258819,-0.965926,0.781831,0.62349,0.5,0.866025,0,0,11,0


Turns out that first iteration's variables: `days_to_quincena` and `is_quincena` are not correlated at all to the target variable so we'll not add them for the second iteration.

In [279]:
# Cell 8: Holiday analysis — testing if Mexican holidays shut down plants
# If holidays matter, we should see volume drop to near-zero on holiday dates.

import holidays

mx_holidays = holidays.MX(years=range(2020, 2027))
df_pre_FE["is_holiday"] = df_pre_FE["programmed_departure_time"].dt.date.isin(mx_holidays).astype(int)

# Compare holiday vs. non-holiday (business hours only, excluding Sundays)
business_no_sunday = df_pre_FE[
    (df_pre_FE["programmed_departure_hour"].between(7, 17)) & 
    (df_pre_FE["day_of_week"] != 6)
].copy()

holiday_comparison = business_no_sunday.groupby("is_holiday")["volume_m3"].agg(["mean", "count"]).reset_index()
holiday_comparison["is_holiday"] = holiday_comparison["is_holiday"].map({0: "Normal day", 1: "Holiday"})

fig = px.bar(
    holiday_comparison,
    x="is_holiday",
    y="mean",
    title="Average Volume: Holidays vs. Normal Days (Business Hours, Excl. Sundays)",
    labels={"mean": "Avg Volume (m³/hr)", "is_holiday": "Day Type"},
    color="is_holiday",
    color_discrete_map={"Normal day": "#1f77b4", "Holiday": "#d62728"}
)
fig.update_layout(width=600, height=400, template="plotly_white", showlegend=False)
fig.show()

print(holiday_comparison)
print(f"\nCorrelation is_holiday with volume: {df_pre_FE['is_holiday'].corr(df_pre_FE['volume_m3']):.3f}")

   is_holiday      mean   count
0  Normal day  9.624678  117525
1     Holiday  1.391035    2541

Correlation is_holiday with volume: -0.050


`is_holiday` justified due to mean decrease 

In [280]:
# Cell 9: Lag feature analysis — testing if past volume predicts future volume
# Lags are the backbone of time-series models. If yesterday's 8am volume 
# correlates with today's 8am volume, we have an autoregressive signal.
# We'll test 24h, 48h, and 1-week lags.

# Create lags PER PLANT (critical: don't mix plants)
df_pre_FE["volume_m3_lag_24h"] = df_pre_FE.groupby("plant_code")["volume_m3"].shift(24)
df_pre_FE["volume_m3_lag_48h"] = df_pre_FE.groupby("plant_code")["volume_m3"].shift(48)
df_pre_FE["volume_m3_lag_1w"] = df_pre_FE.groupby("plant_code")["volume_m3"].shift(168)

# Same for remission_count
df_pre_FE["remission_count_lag_24h"] = df_pre_FE.groupby("plant_code")["remission_count"].shift(24)
df_pre_FE["remission_count_lag_48h"] = df_pre_FE.groupby("plant_code")["remission_count"].shift(48)
df_pre_FE["remission_count_lag_1w"] = df_pre_FE.groupby("plant_code")["remission_count"].shift(168)

# Correlations (drop NaNs from shift)
lag_features = [
    "volume_m3_lag_24h", "volume_m3_lag_48h", "volume_m3_lag_1w",
    "remission_count_lag_24h", "remission_count_lag_48h", "remission_count_lag_1w"
]

lag_corrs = df_pre_FE[lag_features + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=lag_corrs.index,
    y=lag_corrs.values,
    marker_color="#2ca02c"
))
fig.update_layout(
    title="Lag Features: Correlation with Current volume_m3",
    xaxis_tickangle=-45,
    yaxis_title="Pearson r",
    template="plotly_white",
    width=700, height=400
)
fig.show()

print("Lag correlations:")
print(lag_corrs.sort_values(ascending=False).round(3))

Lag correlations:
remission_count_lag_1w     0.490
volume_m3_lag_1w           0.431
remission_count_lag_24h    0.415
volume_m3_lag_24h          0.383
remission_count_lag_48h    0.324
volume_m3_lag_48h          0.297
Name: volume_m3, dtype: float64


Good correlation between lags and target variable, specially remission and volume 1 week lag

In [281]:
# Cell 10: Rolling window analysis — trend and volatility features
# Rolling means capture recent momentum. Rolling std captures volatility.
# High volatility might mean the next hour is harder to predict.
# We'll test 24h, 48h, and 1-week windows for both volume and remission count.

# Rolling means
df_pre_FE["volume_m3_roll_mean_24h"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(lambda x: x.shift(1).rolling(24).mean())
df_pre_FE["volume_m3_roll_mean_48h"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(lambda x: x.shift(1).rolling(48).mean())
df_pre_FE["volume_m3_roll_mean_1w"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(lambda x: x.shift(1).rolling(168).mean())

df_pre_FE["remission_count_roll_mean_24h"] = df_pre_FE.groupby("plant_code")["remission_count"].transform(lambda x: x.shift(1).rolling(24).mean())
df_pre_FE["remission_count_roll_mean_48h"] = df_pre_FE.groupby("plant_code")["remission_count"].transform(lambda x: x.shift(1).rolling(48).mean())
df_pre_FE["remission_count_roll_mean_1w"] = df_pre_FE.groupby("plant_code")["remission_count"].transform(lambda x: x.shift(1).rolling(168).mean())

# Rolling stds
df_pre_FE["volume_m3_roll_std_24h"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(lambda x: x.shift(1).rolling(24).std())
df_pre_FE["volume_m3_roll_std_48h"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(lambda x: x.shift(1).rolling(48).std())
df_pre_FE["volume_m3_roll_std_1w"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(lambda x: x.shift(1).rolling(168).std())

df_pre_FE["remission_count_roll_std_24h"] = df_pre_FE.groupby("plant_code")["remission_count"].transform(lambda x: x.shift(1).rolling(24).std())
df_pre_FE["remission_count_roll_std_48h"] = df_pre_FE.groupby("plant_code")["remission_count"].transform(lambda x: x.shift(1).rolling(48).std())
df_pre_FE["remission_count_roll_std_1w"] = df_pre_FE.groupby("plant_code")["remission_count"].transform(lambda x: x.shift(1).rolling(168).std())

# Correlations
roll_features = [
    "volume_m3_roll_mean_24h", "volume_m3_roll_mean_48h", "volume_m3_roll_mean_1w",
    "volume_m3_roll_std_24h", "volume_m3_roll_std_48h", "volume_m3_roll_std_1w",
    "remission_count_roll_mean_24h", "remission_count_roll_mean_48h", "remission_count_roll_mean_1w",
    "remission_count_roll_std_24h", "remission_count_roll_std_48h", "remission_count_roll_std_1w"
]

roll_corrs = df_pre_FE[roll_features + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")

fig = go.Figure()
colors = ["#1f77b4" if "mean" in f else "#ff7f0e" for f in roll_corrs.index]
fig.add_trace(go.Bar(x=roll_corrs.index, y=roll_corrs.values, marker_color=colors))
fig.update_layout(
    title="Rolling Window Features: Correlation with volume_m3",
    xaxis_tickangle=-45,
    yaxis_title="Pearson r",
    template="plotly_white",
    width=900, height=450
)
fig.show()

print("Rolling correlations:")
print(roll_corrs.sort_values(ascending=False).round(3))

Rolling correlations:
remission_count_roll_mean_24h    0.195
volume_m3_roll_mean_24h          0.190
remission_count_roll_std_24h     0.174
volume_m3_roll_std_24h           0.142
remission_count_roll_mean_48h    0.142
volume_m3_roll_mean_48h          0.140
volume_m3_roll_mean_1w           0.126
remission_count_roll_mean_1w     0.122
remission_count_roll_std_48h     0.120
remission_count_roll_std_1w      0.106
volume_m3_roll_std_48h           0.096
volume_m3_roll_std_1w            0.084
Name: volume_m3, dtype: float64


These rolls are weak but still good for the model

In [282]:
# Cell 11: Operational features — order size trend and plant maturity
# 1. volume_per_remission_7d_avg: If orders are getting bigger/smaller, 
#    the model should know. Bigger orders = fewer remissions but same volume.
# 2. days_since_first_record: Older plants might have more stable patterns.

# Order size = volume / remission_count (avoid division by zero)
# Use np.nan so column stays float64, not object
df_pre_FE["order_size"] = df_pre_FE["volume_m3"] / df_pre_FE["remission_count"].replace(0, np.nan)

df_pre_FE["volume_per_remission_7d_avg"] = df_pre_FE.groupby("plant_code")["order_size"].transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).mean()
)

# Days since first record for this plant
first_record = df_pre_FE.groupby("plant_code")["programmed_departure_time"].transform("min")
df_pre_FE["days_since_first_record"] = (df_pre_FE["programmed_departure_time"] - first_record).dt.days

# Correlations
ops_features = ["volume_per_remission_7d_avg", "days_since_first_record"]
ops_corrs = df_pre_FE[ops_features + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")

fig = go.Figure()
fig.add_trace(go.Bar(x=ops_corrs.index, y=ops_corrs.values, marker_color=["#9467bd", "#8c564b"]))
fig.update_layout(
    title="Operational Features: Correlation with volume_m3",
    yaxis_title="Pearson r",
    template="plotly_white",
    width=500, height=400
)
fig.show()

print("Operational correlations:")
print(ops_corrs.round(3))

# Clean up helper column
df_pre_FE = df_pre_FE.drop(columns=["order_size"])

Operational correlations:
volume_per_remission_7d_avg    0.016
days_since_first_record        0.008
Name: volume_m3, dtype: float64


In [283]:
# Cell 12: Plant activity features
# was_open: Was there any volume in the past 24 hours? 
#   (Or past 7 days — let's test both definitions)
# days_since_last_open: Days since the last hour with volume > 0

# Definition A: was_open = any volume in past 24h
df_pre_FE["was_open_24h"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(
    lambda x: (x.shift(1).rolling(24, min_periods=1).sum() > 0).astype(int)
)

# Definition B: was_open = any volume in past 7 days  
df_pre_FE["was_open_7d"] = df_pre_FE.groupby("plant_code")["volume_m3"].transform(
    lambda x: (x.shift(1).rolling(168, min_periods=1).sum() > 0).astype(int)
)

# Days since last open (hours with volume > 0)
# Convert to "days ago" by finding the last non-zero volume timestamp
def days_since_last_open(group):
    # Boolean series: True where volume > 0
    is_open = group["volume_m3"] > 0
    # Forward-fill the timestamp of last open, then compute days diff
    last_open_time = group["programmed_departure_time"].where(is_open).ffill()
    days_since = (group["programmed_departure_time"] - last_open_time).dt.total_seconds() / 86400
    return days_since

df_pre_FE["days_since_last_open"] = df_pre_FE.groupby("plant_code", group_keys=False).apply(days_since_last_open)

# Correlations
activity_features = ["was_open_24h", "was_open_7d", "days_since_last_open"]
activity_corrs = df_pre_FE[activity_features + ["volume_m3"]].corr()["volume_m3"].drop("volume_m3")

fig = go.Figure()
fig.add_trace(go.Bar(x=activity_corrs.index, y=activity_corrs.values, marker_color=["#17becf", "#bcbd22", "#e377c2"]))
fig.update_layout(
    title="Activity Features: Correlation with volume_m3",
    yaxis_title="Pearson r",
    template="plotly_white",
    width=600, height=400
)
fig.show()

print("Activity feature correlations:")
print(activity_corrs.round(3))

# Also check: what % of rows have was_open_24h=0? Should be ~ Sundays + late night zeros
print(f"\nwas_open_24h = 0: {(df_pre_FE['was_open_24h'] == 0).mean():.1%}")
print(f"was_open_7d = 0: {(df_pre_FE['was_open_7d'] == 0).mean():.1%}")

Activity feature correlations:
was_open_24h            0.170
was_open_7d             0.064
days_since_last_open   -0.121
Name: volume_m3, dtype: float64

was_open_24h = 0: 18.6%
was_open_7d = 0: 2.3%


In [284]:
df_pre_FE.head()    

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,...,volume_m3_roll_std_48h,volume_m3_roll_std_1w,remission_count_roll_std_24h,remission_count_roll_std_48h,remission_count_roll_std_1w,volume_per_remission_7d_avg,days_since_first_record,was_open_24h,was_open_7d,days_since_last_open
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0.0
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,...,NaN,NaN,NaN,NaN,NaN,5.750000,0,1,1,0.0
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,...,NaN,NaN,NaN,NaN,NaN,4.750000,0,1,1,0.0
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,...,NaN,NaN,NaN,NaN,NaN,4.500000,0,1,1,0.0
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,...,NaN,NaN,NaN,NaN,NaN,4.142857,0,1,1,0.0


In [285]:
# Cell 13: Cleanup rejected/intermediate features and export
# All features already exist from cells 2-12. Just clean up and save.

# Drop rejected features + intermediates

df_pre_FE = df_pre_FE.drop(columns=["days_to_quincena", "is_quincena", "was_open_7d", "days_since_first_record"])

# Rename was_open_24h if it wasn't renamed yet
if "was_open_24h" in df_pre_FE.columns:
    df_pre_FE = df_pre_FE.rename(columns={"was_open_24h": "was_open"})

# Final feature inventory (exclude raw/target cols)
exclude = ["programmed_departure_time", "plant_code", "volume_m3", "remission_count", "is_imputed"]
feature_cols = [c for c in df_pre_FE.columns if c not in exclude]

print(f"Final dataset shape: {df_pre_FE.shape}")
print(f"Feature count: {len(feature_cols)}")
print(f"\nFeatures ({len(feature_cols)}):")
for i, c in enumerate(feature_cols, 1):
    n_nan = df_pre_FE[c].isna().sum()
    print(f"  {i:2d}. {c:<35} NaNs: {n_nan:>6,}")

Final dataset shape: (305521, 42)
Feature count: 37

Features (37):
   1. month_of_year                       NaNs:      0
   2. day_of_week                         NaNs:      0
   3. day_of_month                        NaNs:      0
   4. programmed_departure_hour           NaNs:      0
   5. year                                NaNs:      0
   6. hour                                NaNs:      0
   7. time_idx                            NaNs:      0
   8. hour_sin                            NaNs:      0
   9. hour_cos                            NaNs:      0
  10. day_week_sin                        NaNs:      0
  11. day_week_cos                        NaNs:      0
  12. month_sin                           NaNs:      0
  13. month_cos                           NaNs:      0
  14. is_weekend                          NaNs:      0
  15. is_sunday                           NaNs:      0
  16. is_holiday                          NaNs:      0
  17. volume_m3_lag_24h                   NaNs:    1

In [286]:
# Cell 13b: Fix missing features and NaN handling

# Fix volume_per_remission_7d_avg NaNs
# NaNs come from: (1) division by zero, (2) rolling startup
# Fill with global mean (order size when remission_count > 0)
mean_order_size = (df_pre_FE["volume_m3"] / df_pre_FE["remission_count"].replace(0, np.nan)).mean()
df_pre_FE["volume_per_remission_7d_avg"] = df_pre_FE["volume_per_remission_7d_avg"].fillna(mean_order_size)
print(f"Filled {6895} NaNs in volume_per_remission_7d_avg with global mean: {mean_order_size:.2f}")

# Also fill any remaining roll_std NaNs (first window has NaN std due to single value)
roll_std_cols = [c for c in df_pre_FE.columns if "roll_std" in c]
for col in roll_std_cols:
    df_pre_FE[col] = df_pre_FE[col].fillna(0.0)

# Re-check
print(f"\nFinal shape: {df_pre_FE.shape}")
nan_check = df_pre_FE.isna().sum()
print(f"Total NaNs in dataset: {nan_check.sum():,}")
if nan_check.sum() > 0:
    print("Columns with NaNs:")
    print(nan_check[nan_check > 0])
else:
    print("Zero NaNs across all columns!")


Filled 6895 NaNs in volume_per_remission_7d_avg with global mean: 3.08

Final shape: (305521, 42)
Total NaNs in dataset: 5,760
Columns with NaNs:
volume_m3_lag_24h                 144
volume_m3_lag_48h                 288
volume_m3_lag_1w                 1008
remission_count_lag_24h           144
remission_count_lag_48h           288
remission_count_lag_1w           1008
volume_m3_roll_mean_24h           144
volume_m3_roll_mean_48h           288
volume_m3_roll_mean_1w           1008
remission_count_roll_mean_24h     144
remission_count_roll_mean_48h     288
remission_count_roll_mean_1w     1008
dtype: int64


In [287]:
df_pre_FE.head()

,programmed_departure_time,plant_code,volume_m3,remission_count,is_imputed,month_of_year,day_of_week,day_of_month,programmed_departure_hour,year,...,remission_count_roll_mean_1w,volume_m3_roll_std_24h,volume_m3_roll_std_48h,volume_m3_roll_std_1w,remission_count_roll_std_24h,remission_count_roll_std_48h,remission_count_roll_std_1w,volume_per_remission_7d_avg,was_open,days_since_last_open
0,2020-02-04 07:00:00,510,11.5,2,0,2,1,4,7,2020,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,3.078256,0,0.0
1,2020-02-04 08:00:00,510,15.0,4,0,2,1,4,8,2020,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,5.750000,1,0.0
2,2020-02-04 09:00:00,510,20.0,5,0,2,1,4,9,2020,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,4.750000,1,0.0
3,2020-02-04 10:00:00,510,21.5,7,0,2,1,4,10,2020,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,4.500000,1,0.0
4,2020-02-04 11:00:00,510,32.0,9,0,2,1,4,11,2020,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,4.142857,1,0.0


In [288]:
# Cell 14: Fix data leakage in days_since_last_open
# =============================================================================
# PROBLEM: Original computation used the CURRENT hour's volume_m3 to determine
# whether the plant was "open". This means days_since_last_open == 0 perfectly
# correlated with volume_m3 > 0 (correlation = 1.0), creating data leakage.
#
# FIX: Use ONLY past hours (shift(1)) to compute the last open time.
# =============================================================================

def days_since_last_open_fixed(group):
    # shift(1) looks at PREVIOUS hours only — never the current row
    is_open = group["volume_m3"].shift(1) > 0
    last_open_time = group["programmed_departure_time"].where(is_open).ffill()
    days_since = (group["programmed_departure_time"] - last_open_time).dt.total_seconds() / 86400
    # First row of each plant has no past — fill with large sentinel value
    return days_since.fillna(999)

df_pre_FE["days_since_last_open"] = df_pre_FE.groupby("plant_code", group_keys=False).apply(days_since_last_open_fixed)

# Verify: correlation with current volume > 0 should NO LONGER be 1.0
leak_check = (df_pre_FE["days_since_last_open"] == 0).corr(df_pre_FE["volume_m3"] > 0)
print(f"Leakage correlation (should be < 1.0): {leak_check:.4f}")

# Re-save the corrected dataset
output_path = "../data/processed/transformed_dataset_post_FE.xlsx"
df_pre_FE.to_excel(output_path, index=False)
print(f"Saved corrected dataset: {output_path}")

Leakage correlation (should be < 1.0): 0.7803
Saved corrected dataset: ../data/processed/transformed_dataset_post_FE.xlsx
